# 01 — Feature Engineering

Derives per-driver telematics features from `data/trips.csv` and joins them with `data/claims.csv`.

| Feature | Description |
|---|---|
| `avg_speed` | Mean speed across all trips (km/h), via haversine distance ÷ time delta |
| `max_speed` | Peak speed recorded across all trips (km/h) |
| `harsh_braking_count` | Number of 1-second intervals where speed drops > 20 km/h |
| `recklessness_score` | Mean absolute bearing change between consecutive segments (degrees) |
| `trips_recorded` | Total number of distinct trips per driver |

In [1]:
import numpy as np
import pandas as pd

In [2]:
trips  = pd.read_csv("../data/trips.csv",  parse_dates=["timestamp"])
claims = pd.read_csv("../data/claims.csv")

print(f"trips : {trips.shape[0]:,} rows × {trips.shape[1]} cols")
print(f"claims: {claims.shape[0]:,} rows × {claims.shape[1]} cols")
trips.head(3)

trips : 150,000 rows × 4 cols
claims: 50 rows × 4 cols


,customer_id,timestamp,latitude,longitude
0,DRV_001,2024-01-01 13:30:00,45.517713,-73.591359
1,DRV_001,2024-01-01 13:30:01,45.517730,-73.591512
2,DRV_001,2024-01-01 13:30:02,45.517720,-73.591647


## Helper Functions

In [3]:
def haversine_vec(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in metres."""
    R = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def bearing_vec(lat1, lon1, lat2, lon2):
    """Compass bearing from point 1 → point 2 (degrees, 0–360)."""
    lat1_r, lat2_r = np.radians(lat1), np.radians(lat2)
    dlon_r = np.radians(lon2 - lon1)
    x = np.sin(dlon_r) * np.cos(lat2_r)
    y = np.cos(lat1_r) * np.sin(lat2_r) - np.sin(lat1_r) * np.cos(lat2_r) * np.cos(dlon_r)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

## 1. Identify Trip Boundaries

Consecutive GPS records within a trip are exactly 1 second apart.  
A gap > 60 seconds between records for the same driver signals the start of a new trip.

In [4]:
trips = trips.sort_values(["customer_id", "timestamp"]).reset_index(drop=True)

trips["dt_s"] = (
    trips.groupby("customer_id")["timestamp"]
    .diff()
    .dt.total_seconds()
)

# First record of each driver (NaN dt_s) or large gap → new trip
trips["new_trip"] = trips["dt_s"].isna() | (trips["dt_s"] > 60)
trips["trip_num"] = trips.groupby("customer_id")["new_trip"].cumsum()

print("Trips per driver (should all be 10):")
print(trips.groupby("customer_id")["trip_num"].nunique().value_counts().to_string())

Trips per driver (should all be 10):
trip_num
10    50


## 2. Compute Per-Segment Speed and Bearing

For each consecutive GPS pair **within the same trip**:

- **Speed** = `haversine(p_{t-1}, p_t)` metres ÷ 1 s × 3.6 → km/h  
- **Bearing** = compass direction from `p_{t-1}` to `p_t`  
- **Bearing change** = `|bearing_t − bearing_{t-1}|` wrapped to 0–180°  
- **Harsh brake** = 1 if `speed_{t-1} − speed_t > 20 km/h`

In [5]:
seg = trips.groupby(["customer_id", "trip_num"])

# Previous-point coordinates (reset at each trip boundary)
trips["prev_lat"] = seg["latitude"].shift(1)
trips["prev_lon"] = seg["longitude"].shift(1)
valid = trips["prev_lat"].notna()

# --- Speed ---
trips.loc[valid, "dist_m"] = haversine_vec(
    trips.loc[valid, "prev_lat"], trips.loc[valid, "prev_lon"],
    trips.loc[valid, "latitude"], trips.loc[valid, "longitude"],
)
trips.loc[valid, "speed_kmh"] = trips.loc[valid, "dist_m"] / trips.loc[valid, "dt_s"] * 3.6

# --- Bearing ---
trips.loc[valid, "bearing"] = bearing_vec(
    trips.loc[valid, "prev_lat"], trips.loc[valid, "prev_lon"],
    trips.loc[valid, "latitude"], trips.loc[valid, "longitude"],
)

# Absolute bearing change between consecutive segments (0–180°)
trips["prev_bearing"] = seg["bearing"].shift(1)
both_b = trips["bearing"].notna() & trips["prev_bearing"].notna()
delta = (trips.loc[both_b, "bearing"] - trips.loc[both_b, "prev_bearing"] + 180) % 360 - 180
trips.loc[both_b, "bearing_change"] = delta.abs()

# --- Harsh braking ---
trips["prev_speed"] = seg["speed_kmh"].shift(1)
both_s = trips["speed_kmh"].notna() & trips["prev_speed"].notna()
trips.loc[both_s, "speed_drop"] = trips.loc[both_s, "prev_speed"] - trips.loc[both_s, "speed_kmh"]
trips["harsh_brake"] = (trips["speed_drop"] > 20).astype(int)

trips[["customer_id", "trip_num", "speed_kmh", "bearing_change", "harsh_brake"]]\
    .dropna(subset=["speed_kmh"])\
    .head(5)

,customer_id,trip_num,speed_kmh,bearing_change,harsh_brake
1,DRV_001,1,43.450700,NaN,0
2,DRV_001,1,38.076729,15.045283,0
3,DRV_001,1,36.149983,8.065265,0
4,DRV_001,1,37.092898,3.528471,0
5,DRV_001,1,37.707721,25.332122,0


## 3. Aggregate Features per Driver

In [6]:
features = (
    trips.groupby("customer_id")
    .agg(
        avg_speed          =("speed_kmh",      "mean"),
        max_speed          =("speed_kmh",      "max"),
        harsh_braking_count=("harsh_brake",    "sum"),
        recklessness_score =("bearing_change", "mean"),
        trips_recorded     =("trip_num",       "nunique"),
    )
    .reset_index()
)

features[["avg_speed", "max_speed", "recklessness_score"]] = \
    features[["avg_speed", "max_speed", "recklessness_score"]].round(2)

print(f"Feature matrix: {features.shape}")
features.head()

Feature matrix: (50, 6)


,customer_id,avg_speed,max_speed,harsh_braking_count,recklessness_score,trips_recorded
0,DRV_001,75.72,137.43,0,14.68,10
1,DRV_002,64.08,137.32,0,15.26,10
2,DRV_003,70.86,137.40,0,14.99,10
3,DRV_004,64.42,137.31,0,15.51,10
4,DRV_005,66.14,137.31,0,14.81,10


## 4. Join with Claims and Save

In [7]:
df = features.merge(claims, on="customer_id", how="left")

df.to_csv("../data/features.csv", index=False)
print(f"Saved → data/features.csv  ({df.shape[0]} rows × {df.shape[1]} cols)")
df.head()

Saved → data/features.csv  (50 rows × 9 cols)


,customer_id,avg_speed,max_speed,harsh_braking_count,recklessness_score,trips_recorded,num_claims,responsible,total_loss
0,DRV_001,75.72,137.43,0,14.68,10,2,True,True
1,DRV_002,64.08,137.32,0,15.26,10,2,True,False
2,DRV_003,70.86,137.40,0,14.99,10,1,True,True
3,DRV_004,64.42,137.31,0,15.51,10,2,True,False
4,DRV_005,66.14,137.31,0,14.81,10,2,True,False


## 5. Sample Output (first 10 drivers)

In [8]:
df.head(10)

,customer_id,avg_speed,max_speed,harsh_braking_count,recklessness_score,trips_recorded,num_claims,responsible,total_loss
0,DRV_001,75.72,137.43,0,14.68,10,2,True,True
1,DRV_002,64.08,137.32,0,15.26,10,2,True,False
2,DRV_003,70.86,137.40,0,14.99,10,1,True,True
3,DRV_004,64.42,137.31,0,15.51,10,2,True,False
4,DRV_005,66.14,137.31,0,14.81,10,2,True,False
5,DRV_006,60.74,137.37,0,15.12,10,2,True,False
6,DRV_007,80.58,137.38,1,14.66,10,3,False,False
7,DRV_008,70.57,137.30,0,15.04,10,3,False,False
8,DRV_009,56.36,137.34,0,14.81,10,3,True,True
9,DRV_010,64.64,137.39,0,15.30,10,2,True,True


## 6. Descriptive Statistics

In [9]:
feature_cols = ["avg_speed", "max_speed", "harsh_braking_count", "recklessness_score", "trips_recorded"]
df[feature_cols].describe().round(2)

,avg_speed,max_speed,harsh_braking_count,recklessness_score,trips_recorded
count,50.00,50.00,50.00,50.00,50.0
mean,41.41,82.55,0.06,7.76,10.0
std,19.62,37.88,0.24,5.13,0.0
min,18.19,50.69,0.00,2.05,10.0
25%,26.05,50.84,0.00,3.43,10.0
50%,33.78,65.25,0.00,6.23,10.0
75%,57.98,137.31,0.00,14.68,10.0
max,85.20,137.46,1.00,16.03,10.0


### Feature Averages by Risk Profile

Sanity check — risky drivers (DRV_001–015) should score higher across all features.

In [10]:
def assign_profile(cid):
    n = int(cid.split("_")[1])
    if n <= 15: return "risky"
    if n <= 40: return "safe"
    return "medium"

df["profile"] = df["customer_id"].apply(assign_profile)
df.groupby("profile")[feature_cols + ["num_claims"]].mean().round(2)

,avg_speed,max_speed,harsh_braking_count,recklessness_score,trips_recorded,num_claims
profile,,,,,,
medium,40.11,79.67,0.0,7.13,10.0,1.20
risky,68.41,137.37,0.2,15.10,10.0,2.13
safe,25.72,50.82,0.0,3.60,10.0,0.48
